In [1]:
import os
import glob
import warnings
import concurrent.futures
import numpy as np
import matplotlib.pyplot as plt
import glob
import pandas as pd

In [2]:
from astropy.io import fits
from astropy import units as u
from astropy.utils.exceptions import AstropyWarning
from astropy.stats import sigma_clipped_stats, sigma_clip

In [3]:
from astropop.image import imcombine, processing, imarith
from astropop.framedata import read_framedata
from astropop.file_collection import FitsFileGroup
from astropop.polarimetry import SLSDualBeamPolarimetry

In [4]:
from photutils.detection import DAOStarFinder
from photutils.centroids import centroid_com
from photutils.aperture import CircularAperture, CircularAnnulus, aperture_photometry

/home/likeableng/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
from collections import defaultdict
from matplotlib.patches import Circle

In [6]:
#Caminho até chegar chegar antes na pasta da noite
CAMINHO_PASTA__REDUCAO = '/home/likeableng/BeACoN/arquivos_de_reducao/'

In [7]:
#Colocar os dados da noite
DATA = "20250719" #ex: "AAAAMMDD"
NOME_ESTRELA = "paqr" #ex: "paqr" "osco"

In [8]:
#Ajuste data
mes = int(DATA[4:6])

nome_mes = ['', 'jan', 'fev', 'mar', 'abr', 'mai', 'jun', 'jul', 
            'ago', 'set', 'out', 'nov', 'dez']

data_ajustada = DATA[2:4]+nome_mes[mes]+DATA[6:]

In [10]:
#Caminhos para pastas reduced e raw

raw_dir = f"{CAMINHO_PASTA__REDUCAO}{data_ajustada}/raw/"

reduced_dir = f"{CAMINHO_PASTA__REDUCAO}{data_ajustada}/reduced/"
os.makedirs(reduced_dir, exist_ok=True)

In [11]:
#Caminho para dados estrela
pasta_raw_estrela = f'{raw_dir}{NOME_ESTRELA}'
pasta_reduced_estrela = f'{reduced_dir}{NOME_ESTRELA}'
os.makedirs(pasta_reduced_estrela, exist_ok=True)

#Caminho para dados calib
pasta_raw_calib = f'{raw_dir}calib/'
pasta_reduced_calib = f'{reduced_dir}calib/'
os.makedirs(pasta_reduced_calib, exist_ok=True)

In [17]:
padrao_busca = os.path.join(pasta_raw_estrela, f'{NOME_ESTRELA}_b_0*.fits')
arquivos_encontrados = sorted(glob.glob(padrao_busca))

In [13]:
valores_lamina = []

In [ ]:
for arq in arquivos_encontrados:
    try:
        hdr = fits.getheader(arq)

        # O valor -999.0 serve como aviso caso a palavra-chave não exista
        valor = hdr.get('LAMINA', -999.0)

        valor = valor[1:]

        if valor != -999.0:
            valores_lamina.append(valor)
            
    except Exception as e:
        print(f"Erro ao ler o ficheiro {os.path.basename(arq)}: {e}")

if valores_lamina:
    valor_mais_alto = max(valores_lamina)
    vezes_repetido = valores_lamina.count(valor_mais_alto)
    
    print("=== RESULTADOS DA LÂMINA ===")
    print(f"Total de leituras bem-sucedidas: {len(valores_lamina)}")
    print(f"Valor mais alto encontrado: {valor_mais_alto}")
    print(f"O valor mais alto aparece {vezes_repetido} vez(es) nas imagens.")
else:
    print("Nenhum valor de lâmina válido foi encontrado nos ficheiros.")


=== RESULTADOS DA LÂMINA ===
Total de leituras bem-sucedidas: 16
Valor mais alto encontrado: B
O valor mais alto aparece 2 vez(es) nas imagens.


In [22]:
if valor_mais_alto == "B":
    numero_de_arquivos_por_pasta = 8

else:
    numero_de_arquivos_por_pasta = 16


In [19]:
import shutil

In [27]:
numero_total_de_imagens = numero_de_arquivos_por_pasta * vezes_repetido
i=0 #Sei que da pra deixar aqui como um e fazer ser menor ou igual
    #, mas por algum motivo tava demorando mais de 30 seg
while i < vezes_repetido:
    i=i+1 
    os.makedirs(f"{pasta_raw_estrela}/Obs_{i}_{NOME_ESTRELA}", exist_ok=True)

    os.makedirs(f"{pasta_reduced_estrela}/Obs_{i}_{NOME_ESTRELA}", exist_ok=True)

In [ ]:
for nome_item in os.listdir(pasta_raw_estrela):
    caminho_origem_completo = os.path.join(pasta_raw_estrela, nome_item)

    if os.path.isfile(caminho_origem_completo):
    
        partes = nome_item.split('_')

        if len(partes) == 3:

            final_do_item = partes[2]
            valor_do_item = os.path.splitext(final_do_item)[0]  
            numero_do_item = int(valor_do_item)

            for i in range(vezes_repetido):
                limite_inferior = i * numero_de_arquivos_por_pasta
                limite_superior = (i + 1) * numero_de_arquivos_por_pasta
                
                if limite_inferior < numero_do_item <= limite_superior:
                    
                    pasta_raw_estrela_i = f"{pasta_raw_estrela}/Obs_{i+1}_{NOME_ESTRELA}"
                    
                    if not os.path.exists(pasta_raw_estrela_i):
                        print(f"A pasta {pasta_raw_estrela_i} não existe")
                        break
                        
                    caminho_destino_completo = os.path.join(pasta_raw_estrela_i, nome_item)
                    shutil.move(caminho_origem_completo, caminho_destino_completo)
                    
                    break

In [51]:
calculo = 8 // 8

print(calculo)

1
